<a href="https://colab.research.google.com/github/vcellmike/PatternsFormation/blob/main/Working/2024_08_21_XGBoost_on_ImageJ_Features.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
import json
import os
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from sklearn.metrics import accuracy_score, classification_report
from transformers import AutoModel, AutoTokenizer, get_scheduler
from torch.utils.data import Dataset, DataLoader, RandomSampler, SequentialSampler
from torch.optim import AdamW
from tqdm.notebook import tqdm, trange
from time import perf_counter
from PIL import Image
import pandas as pd
#from google.colab import drive
from PIL import Image
import pandas as pd
import os
import numpy as np
import matplotlib.pyplot as plt
import pickle

print("All dependencies present.")

All dependencies present.


In [14]:
# make a dataframe with the names and feat_dfs and inverse_feat_dfs:
real_imj_df = {"path":[],"feats_df":[],"inverse_feats_df":[]}

reg_paths = os.listdir("content/ImageJ_data/Inverse_resized/")

for path in reg_paths:
  if path != ".ipynb_checkpoints":
    real_imj_df["path"].append(path)
    real_imj_df["feats_df"].append(pd.read_csv("content/ImageJ_data/Regular_resized/" + path))
    real_imj_df["inverse_feats_df"].append(pd.read_csv("content/ImageJ_data/Inverse_resized/" + path))

real_df = pd.DataFrame(real_imj_df)

In [15]:
real_df

path  \
0     LF10_11-20-23_(green)_(0-152).tif.csv   
1     mpar_rto(c-c)_(green)_(0-137).tif.csv   
2   mpar_rto_crispr_(green)_(0-155).tif.csv   
3   MLC_F1_11-20-23_(green)_(0-105).tif.csv   
4      RTO_rnai_med_(green)_(0-109).tif.csv   
5      NEGAN_crispr_(green)_(0-150).tif.csv   
6           mpar_wt_(green)_(0-170).tif.csv   
7      RTO_rnai_low_(green)_(0-115).tif.csv   
8        bHLH2_RNAi_(green)_(0-170).tif.csv   
9     RTO_rnai_high_(green)_(0-103).tif.csv   
10            F2_260_green)_(0-105).tif.csv   
11           F2_A 8_(green)_(0-100).tif.csv   
12   bHLH2_OE_(green)_(0-141_thresh.tif.csv   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    

In [16]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('display.max_colwidth', None) 
real_df.shape

columns0 = ['feats_df', 'inverse_feats_df']

real_df2 = real_df.drop(columns=columns0)

real_df2

#real_df.head(14)

,path
0,LF10_11-20-23_(green)_(0-152).tif.csv
1,mpar_rto(c-c)_(green)_(0-137).tif.csv
2,mpar_rto_crispr_(green)_(0-155).tif.csv
3,MLC_F1_11-20-23_(green)_(0-105).tif.csv
4,RTO_rnai_med_(green)_(0-109).tif.csv
5,NEGAN_crispr_(green)_(0-150).tif.csv
6,mpar_wt_(green)_(0-170).tif.csv
7,RTO_rnai_low_(green)_(0-115).tif.csv
8,bHLH2_RNAi_(green)_(0-170).tif.csv
9,RTO_rnai_high_(green)_(0-103).tif.csv


In [17]:
# Add regular features

# first list is mean, second list is std except for mean and median
feats = {'Mean':[],'Median':[],'Area':[[],[]], 'X':[[],[]], 'Y':[[],[]], 'Perim.':[[],[]], 'BX':[[],[]],
         'BY':[[],[]], 'Width':[[],[]], 'Height':[[],[]], 'Major':[[],[]], 'Minor':[[],[]],
       'Angle':[[],[]], 'Circ.':[[],[]], 'Feret':[[],[]], 'IntDen':[[],[]], '%Area':[[],[]],
       'RawIntDen':[[],[]], 'FeretX':[[],[]], 'FeretY':[[],[]], 'FeretAngle':[[],[]], 'MinFeret':[[],[]], 'AR':[[],[]],
       'Round':[[],[]], 'Solidity':[[],[]]}


counter = 0

# iterate through all sims
for n in range(real_df.shape[0]): # grab the feats dataframe for this sim
  fdf = real_df["feats_df"][n] #iterate through all feats within the feats dictionary

  for i in range(len(feats.keys())): # select the current feat
    feat = list(feats.keys())[i]

    # if feat in ['Mean_inverted','Median_inverted']: # check for feats that should be added from the overall measurement (first row)
    if feat in ['Mean','Median']: # check for feats that should be added from the overall measurement (first row)
      # feats[feat].append(fdf[feat[0:-9]][0]) #inverse
      feats[feat].append(fdf[feat[:]][0])
      # feats[feat].append(fdf[feat][0])

    else:
      if fdf.shape[0] <= 1: # check if the fdf is only one measurement (empty result)
        feats[feat][0].append(np.mean(np.array(fdf[feat][0]))) # mean of sole measurement
        feats[feat][1].append(np.std(np.array(fdf[feat][0]))) #std of sole measurement (0)

      else:
        feats[feat][0].append(np.mean(np.array(fdf[feat][1:]))) # add mean for each of the rest of the feats using the remaining rows
        feats[feat][1].append(np.std(np.array(fdf[feat][1:]))) # add std for each of the rest of the feats using the remaining rows

  if counter % 1000 == 0:
    print(counter)
  counter += 1

print("Counting done")

0
Counting done


In [18]:
num_spots = []

for i in range(real_df.shape[0]):
  fdf = real_df["feats_df"][i] #iterate through all feats within the feats dictionary
  if fdf.shape[0] <= 1:
    num_spots.append(0)
  else:
    num_spots.append(fdf.shape[0]-1)

# feats_df["num_spots_inverted"] = num_spots
real_df["num_spots"] = num_spots

In [19]:
for key in list(feats.keys()):
  # if key in ['Mean_inverted','Median_inverted']:
  if key in ['Mean','Median']:
    real_df[key] = feats[key]
  else:
    # feats_df[key + "_inverted_mean"] = feats[key][0]
    # feats_df[key + "_inverted_std"] = feats[key][1]
    real_df[key + "_mean"] = feats[key][0]
    real_df[key + "_std"] = feats[key][1]

In [20]:
# Add regular features

# first list is mean, second list is std except for mean and median
# feats = {'Mean':[],'Median':[],'Area':[[],[]], 'X':[[],[]], 'Y':[[],[]], 'Perim.':[[],[]], 'BX':[[],[]],
#          'BY':[[],[]], 'Width':[[],[]], 'Height':[[],[]], 'Major':[[],[]], 'Minor':[[],[]],
#        'Angle':[[],[]], 'Circ.':[[],[]], 'Feret':[[],[]], 'IntDen':[[],[]], '%Area':[[],[]],
#        'RawIntDen':[[],[]], 'FeretX':[[],[]], 'FeretY':[[],[]], 'FeretAngle':[[],[]], 'MinFeret':[[],[]], 'AR':[[],[]],
#        'Round':[[],[]], 'Solidity':[[],[]]}

feats = {'Mean_inverted':[],'Median_inverted':[],'Area':[[],[]], 'X':[[],[]], 'Y':[[],[]], 'Perim.':[[],[]], 'BX':[[],[]],
         'BY':[[],[]], 'Width':[[],[]], 'Height':[[],[]], 'Major':[[],[]], 'Minor':[[],[]],
       'Angle':[[],[]], 'Circ.':[[],[]], 'Feret':[[],[]], 'IntDen':[[],[]], '%Area':[[],[]],
       'RawIntDen':[[],[]], 'FeretX':[[],[]], 'FeretY':[[],[]], 'FeretAngle':[[],[]], 'MinFeret':[[],[]], 'AR':[[],[]],
       'Round':[[],[]], 'Solidity':[[],[]]}


counter = 0

# iterate through all sims
for n in range(real_df.shape[0]): # grab the feats dataframe for this sim
  fdf = real_df["inverse_feats_df"][n] #iterate through all feats within the feats dictionary

  for i in range(len(feats.keys())): # select the current feat
    feat = list(feats.keys())[i]

    if feat in ['Mean_inverted','Median_inverted']: # check for feats that should be added from the overall measurement (first row)
    # if feat in ['Mean','Median']: # check for feats that should be added from the overall measurement (first row)
      feats[feat].append(fdf[feat[0:-9]][0]) #inverse
      # feats[feat].append(fdf[feat[:]][0])

    else:
      if fdf.shape[0] <= 1: # check if the fdf is only one measurement (empty result)
        feats[feat][0].append(np.mean(np.array(fdf[feat][0]))) # mean of sole measurement
        feats[feat][1].append(np.std(np.array(fdf[feat][0]))) #std of sole measurement (0)

      else:
        feats[feat][0].append(np.mean(np.array(fdf[feat][1:]))) # add mean for each of the rest of the feats using the remaining rows
        feats[feat][1].append(np.std(np.array(fdf[feat][1:]))) # add std for each of the rest of the feats using the remaining rows

  if counter % 1000 == 0:
    print(counter)
  counter += 1

print("Counter complete")

0
Counter complete


In [21]:
num_spots = []

for i in range(real_df.shape[0]):
  fdf = real_df["inverse_feats_df"][i] #iterate through all feats within the feats dictionary
  if fdf.shape[0] <= 1:
    num_spots.append(0)
  else:
    num_spots.append(fdf.shape[0]-1)

real_df["num_spots_inverted"] = num_spots
# real_df["num_spots"] = num_spots

In [22]:
for key in list(feats.keys()):
  if key in ['Mean_inverted','Median_inverted']:
  # if key in ['Mean','Median']:
    real_df[key] = feats[key]
  else:
    real_df[key + "_inverted_mean"] = feats[key][0]
    real_df[key + "_inverted_std"] = feats[key][1]
    # real_df[key + "_mean"] = feats[key][0]
    # real_df[key + "_std"] = feats[key][1]

In [23]:
print(real_df.columns[3:101])

Index(['num_spots', 'Mean', 'Median', 'Area_mean', 'Area_std', 'X_mean', 'X_std', 'Y_mean', 'Y_std', 'Perim._mean', 'Perim._std', 'BX_mean', 'BX_std', 'BY_mean', 'BY_std', 'Width_mean', 'Width_std', 'Height_mean', 'Height_std', 'Major_mean', 'Major_std', 'Minor_mean', 'Minor_std', 'Angle_mean', 'Angle_std', 'Circ._mean', 'Circ._std', 'Feret_mean', 'Feret_std', 'IntDen_mean', 'IntDen_std', '%Area_mean', '%Area_std', 'RawIntDen_mean', 'RawIntDen_std', 'FeretX_mean', 'FeretX_std', 'FeretY_mean', 'FeretY_std', 'FeretAngle_mean', 'FeretAngle_std', 'MinFeret_mean', 'MinFeret_std', 'AR_mean', 'AR_std', 'Round_mean', 'Round_std', 'Solidity_mean', 'Solidity_std', 'num_spots_inverted', 'Mean_inverted', 'Median_inverted', 'Area_inverted_mean', 'Area_inverted_std', 'X_inverted_mean', 'X_inverted_std', 'Y_inverted_mean', 'Y_inverted_std', 'Perim._inverted_mean', 'Perim._inverted_std', 'BX_inverted_mean', 'BX_inverted_std', 'BY_inverted_mean', 'BY_inverted_std', 'Width_inverted_mean',
       'Width_

In [24]:
real_df.to_pickle("./data/real_df_xgboost.pkl")